<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 27 · ML for Portfolio and Risk Signals

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch27_ml_portfolio_risk_signals.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This chapter shows how machine learning can support portfolio risk monitoring without automating judgment. You will learn to build timestamped risk features such as credit spread levels in basis points and volatility proxies in percent, to construct a transparent threshold rule as a baseline alert, and to combine several features into a fixed logistic-style stress score. You will compare the two alert mechanisms through precision, recall, and false-positive counts, weigh the asymmetric costs of missed stress against wasted attention, and design a human-in-the-loop workflow in which every alert carries Python-verified evidence and a documented decision.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script applies two transparent alert mechanisms to a 12-row weekly synthetic panel: a threshold rule that fires when the spread exceeds 150 bp or volatility exceeds 18 percent, and a fixed, hand-specified score passed through a sigmoid with an alert threshold of 0.55. It prints the weekly scores with both alert columns and a descriptive precision/recall table for each method. The score is a teaching formula, not a fitted or calibrated model, and all error counts are in-sample.

In [ ]:
SCRIPT = ROOT / "code" / "ch27_ml_portfolio_risk_signals.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Check both alert definitions against the output: the rule fires on spread above 150 bp or volatility above 18 percent; the score alert fires at 0.55.
Identify the stress dates each method misses: the rule misses 2026-01-23 (spread 145 bp, volatility 14 percent), while the fixed score also misses 2026-01-30.
Compare the printed metrics: rule precision 0.857 and recall 0.857, fixed score precision 1.000 and recall 0.714, and locate the rule's single false positive on 2026-03-06.
Confirm the units: spread in bp, volatility in percent, and the score a unitless sigmoid value that is not a calibrated probability.
State the evidence limit: the counts describe the same 12 synthetic weeks used to define the alerts, so they are not out-of-sample performance.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch27_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Combine two signals into a capped position tilt

This lab isolates one mechanism: combining two risk features into a single score, mapping the score to a smooth portfolio tilt, and capping the resulting position change. The inputs are three illustrative weekly observations; the position is stated in USD million, and the cap binds in the stressed weeks.

In [ ]:
import numpy as np
import pandas as pd

weeks = pd.DataFrame({  # three illustrative weekly observations
    "week": ["2026-04-03", "2026-04-10", "2026-04-17"],
    "spread": [120, 165, 230],  # credit spread, bp
    "volatility": [11.0, 17.0, 26.0],  # volatility proxy, percent
})
score = -8.0 + 0.03 * weeks.spread + 0.18 * weeks.volatility  # combined
weeks["tilt"] = np.tanh(score)  # smooth tilt in [-1, 1]
base, cap = 100.0, 15.0  # position in USD million, cap on change
weeks["change"] = np.clip(weeks.tilt * 25.0, -cap, cap)
weeks["target"] = base + weeks.change
weeks.round(3)

### What to observe

Expected output: tilt scores -0.984, 0.010, and 0.998, giving uncapped position changes of -24.61, 0.25, and 24.96 USD million. The 15 USD million cap binds in the first and third weeks, so the targets are 85.0, 100.25, and 115.0 USD million. The signal is unbounded, but the position change is not; the cap keeps a strong signal from over-sizing the trade.

## Exercises

1. **Rule audit.** Recompute the rule alerts for all 12 weeks and confirm the single false positive on 2026-03-06. *Hint:* The spread exceeds 150 bp on that date even though the stress label is 0.
2. **Missed stress.** Identify the stress dates that each method misses and explain the misses in terms of the inputs. *Hint:* On 2026-01-23, spread 145 bp and volatility 14 percent sit below both rule thresholds, and the fixed score is only 0.244.
3. **Threshold trade-off.** Lower the score alert threshold from 0.55 to 0.40 and recompute precision and recall for both methods. *Hint:* Only the score column changes; the rule stays fixed, and 2026-01-30 should now be captured.
4. **Signal to action.** Extend the mini-lab so the capped tilt feeds a short review note that cites the week, spread, volatility, and score. *Hint:* The chapter requires generated risk notes to cite verified Python numbers and never to instruct trades automatically.
5. **Cost asymmetry.** Write one paragraph on which error, false positive or false negative, is more expensive for a risk committee and how that should set the alert threshold. *Hint:* The answer depends on team capacity and portfolio exposure, not on the metrics table alone.

For more detail, see Chapter 27 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">